# Experiment 2: ACS2023 income target with Adult and ACS2014 baseline MIA
Retain the restored Experiment 2 target: 145,000 training records, MLP hidden
layers 512â†’256â†’128, 100 epochs, batch size 256, learning rate .001, no dropout
or weight decay, seed 42, final weights retained. Target data changes to California ACS2023.
Each of validation, test, MIA members and MIA nonmembers has 10,000 records.
Keep the original ten predictor concepts. ACS2023 uses native RELSHIPP relationship
categories where ACS2018/2014 used RELP. Each model fits its own native encoding;
no equivalence between old and new relationship category codes is assumed.
The other nine fields remain AGEP, COW, SCHL, MAR, OCCP, POBP, WKHP, SEX and RAC1P.
Income label is nominal PINCP >$50,000; retain original ACSIncome eligibility.

Compare two separate auxiliary attacks: UCI Adult and California ACS2014.
Each uses eight shadows cycling the original five algorithms: XGBoost, random forest, MLP, decision tree,
logistic regression, retaining their original settings, seeds and 40/20/40 splits.
Extract 2,000 known members and 2,000 nonmembers per shadow. Group attack splits
by auxiliary record ID and choose thresholds on auxiliary validation only.
Both attacks use logit, blind walk, entropy and cross-entropy with the same RF.
The ACS2014 attack reuses the same 20,000 target feature rows as the Adult attack.
This notebook is prepared for review. Ask the user before executing cells.


In [1]:
from pathlib import Path
import json
import joblib
import numpy as np
import torch
torch.set_num_threads(4)
from IPython.display import display
import importlib
import pandas as pd
import src.data
import src.training
importlib.reload(src.data)
importlib.reload(src.training)
from src.data import split_acs_data, subset_target_training
from src.exp2_year_baseline import (load_baseline_year, features_for_year,
    preprocess_partitions, train_acs_shadows)
from src.training import train_target_model, evaluate_model

OVERFIT_SEED = 42
OVERFIT_TRAIN_SIZE = 145_000
SPLIT_CONFIG = dict(validation_size=10_000, test_size=10_000,
                   mia_nonmember_size=10_000, mia_member_size=10_000, seed=OVERFIT_SEED)
OVERFIT_CONFIG = dict(
    hidden_dims=(512, 256, 128), dropout=0.0, epochs=100,
    batch_size=256, learning_rate=1e-3, weight_decay=0.0,
    seed=OVERFIT_SEED, device='cpu', early_stopping=False,
    restore_best=False, monitor_train=True,
)
N_SHADOW_MODELS = 8
RUN_DIR = Path('artifacts/exp2_2023_adult_2014_baseline_8shadows')
RUN_DIR.mkdir(parents=True,exist_ok=True)
TARGET_FEATURES = features_for_year(2023)
assert len(TARGET_FEATURES)==10 and 'RELSHIPP' in TARGET_FEATURES and 'RELP' not in TARGET_FEATURES
print('Target predictors:',TARGET_FEATURES)


Target predictors: ['AGEP', 'COW', 'SCHL', 'MAR', 'OCCP', 'POBP', 'RELSHIPP', 'WKHP', 'SEX', 'RAC1P']


In [2]:
overfit_source = load_baseline_year(2023)
baseline_partitions_for_ids = split_acs_data(overfit_source, **SPLIT_CONFIG)
overfit_partitions = subset_target_training(
    baseline_partitions_for_ids, train_size=OVERFIT_TRAIN_SIZE, seed=OVERFIT_SEED)
print('Unused baseline training records:',
      len(baseline_partitions_for_ids['train']) - len(overfit_partitions['train']))
pd.DataFrame({name: {'records': len(frame), 'positive_rate': frame['income_gt_50k'].mean()}
              for name, frame in overfit_partitions.items()}).T
assert len(overfit_partitions['train'])==OVERFIT_TRAIN_SIZE
assert set(overfit_partitions['mia_members'].index)<=set(overfit_partitions['train'].index)
for name in ['validation','test','mia_nonmembers']:
    assert set(overfit_partitions[name].index).isdisjoint(overfit_partitions['train'].index)


Unused baseline training records: 28278


In [3]:
# Fit ONLY on the reduced training set. Do not reuse the baseline encoder.
overfit_preprocessor, overfit_arrays = preprocess_partitions(overfit_partitions,TARGET_FEATURES)
{name: values['X'].shape for name, values in overfit_arrays.items()}

{'validation': (10000, 818),
 'test': (10000, 818),
 'mia_nonmembers': (10000, 818),
 'train': (145000, 818),
 'mia_members': (10000, 818)}

In [4]:
# This cell trains when YOU run it. A large CPU model may take time.
overfit_model, overfit_history, overfit_metadata = train_target_model(
    overfit_arrays['train']['X'], overfit_arrays['train']['y'],
    overfit_arrays['validation']['X'], overfit_arrays['validation']['y'],
    **OVERFIT_CONFIG)
print('Returned epoch:', overfit_metadata['returned_epoch'])
pd.DataFrame(overfit_history)
from src.training import save_target_model
save_target_model(overfit_model,overfit_metadata,RUN_DIR/'target_model.pt')
joblib.dump(overfit_preprocessor,RUN_DIR/'target_preprocessor.joblib')


Epoch 001: train loss=0.4149, validation loss=0.3980, validation accuracy=0.8204


Epoch 002: train loss=0.3879, validation loss=0.4022, validation accuracy=0.8164


Epoch 003: train loss=0.3738, validation loss=0.4005, validation accuracy=0.8196


Epoch 004: train loss=0.3534, validation loss=0.4023, validation accuracy=0.8179


Epoch 005: train loss=0.3286, validation loss=0.4210, validation accuracy=0.8161


Epoch 006: train loss=0.2999, validation loss=0.4486, validation accuracy=0.8093


... (91 outputs left out)

Epoch 098: train loss=0.0212, validation loss=2.0162, validation accuracy=0.7815


Epoch 099: train loss=0.0206, validation loss=1.9643, validation accuracy=0.7851


Epoch 100: train loss=0.0224, validation loss=1.9818, validation accuracy=0.7836
Returned epoch: 100


['artifacts\\exp2_2023_adult_2014_baseline_8shadows\\target_preprocessor.joblib']

## Inspect overfitting
Compare end-of-epoch training accuracy/loss against validation. train_loss is averaged across updates during the epoch; train_eval_log_loss evaluates the final weights for that epoch, so it is the directly comparable loss. A growing gap supports overfitting; a high training accuracy alone does not.

In [5]:
overfit_learning_curve = pd.DataFrame(overfit_history)
overfit_learning_curve['accuracy_gap_pp'] = 100 * (
    overfit_learning_curve['train_eval_accuracy'] - overfit_learning_curve['validation_accuracy'])
overfit_learning_curve['validation_minus_train_loss'] = (
    overfit_learning_curve['validation_log_loss'] - overfit_learning_curve['train_eval_log_loss'])
overfit_learning_curve[['epoch', 'train_eval_accuracy', 'validation_accuracy',
    'accuracy_gap_pp', 'train_eval_log_loss', 'validation_log_loss',
    'validation_minus_train_loss']].iloc[::10].copy()

,epoch,train_eval_accuracy,validation_accuracy,accuracy_gap_pp,train_eval_log_loss,validation_log_loss,validation_minus_train_loss
0,1,0.825455,0.8204,0.505517,0.386007,0.397961,0.011953
10,11,0.940372,0.7913,14.907241,0.147582,0.810856,0.663274
20,21,0.974476,0.7778,19.667586,0.068081,1.374979,1.306898
30,31,0.982269,0.7846,19.766897,0.044428,1.612187,1.567758
40,41,0.985655,0.7777,20.795517,0.037234,1.625761,1.588527
50,51,0.988359,0.7879,20.045862,0.028105,1.786201,1.758096
60,61,0.988400,0.7786,20.980000,0.028152,1.798832,1.770680
70,71,0.990152,0.7863,20.385172,0.024011,1.877426,1.853415
80,81,0.991634,0.7844,20.723448,0.019690,1.896257,1.876566
90,91,0.991179,0.7833,20.787931,0.020843,1.904386,1.883543


## Final income-prediction evaluation
Evaluate after the fixed training budget. Do not choose epochs/settings using test results. The membership attack below uses this target and its own fitted preprocessor.


In [6]:
overfit_metrics = {name: evaluate_model(overfit_model, values['X'], values['y'])
    for name, values in overfit_arrays.items() if name in ['train', 'validation', 'test']}
display(pd.DataFrame(overfit_metrics).T)
print('Train-validation accuracy gap (percentage points):',
      100 * (overfit_metrics['train']['accuracy'] - overfit_metrics['validation']['accuracy']))
print('Train-test accuracy gap (percentage points):',
      100 * (overfit_metrics['train']['accuracy'] - overfit_metrics['test']['accuracy']))

,accuracy,roc_auc,log_loss
validation,0.783600,0.850963,1.981811
test,0.784600,0.852023,2.020234
train,0.991614,0.999767,0.019786


Train-validation accuracy gap (percentage points): 20.801379310344835
Train-test accuracy gap (percentage points): 20.701379310344837


## Random forest membership attack: complete-feature baseline
Run after training the overfit 2023 target. This uses the same attack procedure as the overfit notebook: eight diverse Adult shadows with original default settings, four paper-inspired features (class-scaled logits, blind walk, entropy, cross-entropy), and a random forest membership classifier.

The attack is fitted on Adult shadow outputs only. Adult attack rows are split by record ID; thresholds are selected on Adult validation only. Feature extraction samples 2,000 members and 2,000 non-members per shadow. Target evaluation uses all 10,000 overfit target MIA members and 10,000 non-members, with this target's own fitted preprocessor. The Adult and ACS2014 auxiliary classifiers each evaluate the same target candidates in this notebook.

Walk defaults: 20 steps, two walks, categorical change probability .05, batch size 512, seed 42. Repeated model queries can take time. The tabular walk is an adaptation, not an exact reproduction; see data/paper_attack_features.md. True income labels are assumed known. No target MIA results select settings. Run cells manually; none were executed during this edit.

In [7]:
import importlib
import src.shadow_models
import src.membership_attack
import src.paper_attack_features
importlib.reload(src.shadow_models)
importlib.reload(src.membership_attack)
importlib.reload(src.paper_attack_features)
from src.shadow_models import load_adult_data, train_shadow_models, summarize_shadow_models
from src.membership_attack import train_membership_classifier, evaluate_attack
from src.paper_attack_features import collect_shadow_paper_features, collect_target_paper_features

WALK_CONFIG = dict(max_steps=20, repeats=2, categorical_change_probability=0.05,
                   batch_size=512, seed=42)
SHADOW_RECORDS_PER_CLASS = 2000
# Train the eight Adult shadows when you run this cell.
# Skip this cell if shadow_results is already loaded and these imports/settings are set.
adult_auxiliary = load_adult_data()
shadow_results = train_shadow_models(adult_auxiliary, seed=42, n_models=N_SHADOW_MODELS)
summarize_shadow_models(shadow_results)
assert len(shadow_results)==N_SHADOW_MODELS


Training shadow 1/8: xgboost


Training shadow 2/8: random_forest


Training shadow 3/8: mlp


Epoch 001: train loss=0.4162, validation loss=0.3245, validation accuracy=0.8483


Epoch 002: train loss=0.3166, validation loss=0.3166, validation accuracy=0.8531


Epoch 003: train loss=0.3112, validation loss=0.3153, validation accuracy=0.8525


... (27 outputs left out)

Epoch 012: train loss=0.2955, validation loss=0.3166, validation accuracy=0.8528


Epoch 013: train loss=0.2941, validation loss=0.3180, validation accuracy=0.8532


Epoch 014: train loss=0.2927, validation loss=0.3155, validation accuracy=0.8517


Epoch 015: train loss=0.2906, validation loss=0.3147, validation accuracy=0.8507


In [8]:
# Extract the four features by querying this notebook's fitted shadows.
adult_for_rf_queries = load_adult_data()
shadow_attack_data, rf_shadow_walk_diagnostics = collect_shadow_paper_features(
    shadow_results, adult_for_rf_queries, per_class=SHADOW_RECORDS_PER_CLASS,
    config=WALK_CONFIG, seed=42)
display(pd.DataFrame(rf_shadow_walk_diagnostics).T)
attack_result = train_membership_classifier(
    shadow_attack_data, seed=42, algorithm='random_forest', feature_set='paper')
# Adult attack diagnostics, not target membership performance.
pd.DataFrame(attack_result['metrics']).T

Attack feature extraction: 512/4,000


Attack feature extraction: 1,024/4,000


Attack feature extraction: 1,536/4,000


Attack feature extraction: 2,048/4,000


Attack feature extraction: 2,560/4,000


Attack feature extraction: 3,072/4,000


... (56 outputs left out)

Attack feature extraction: 3,584/4,000


Attack feature extraction: 4,000/4,000


,record_queries,max_record_queries,walk_censored_fraction,blind_walk_enabled,config
xgboost_01,130632,164000,0.685625,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
random_forest_02,138296,164000,0.769,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
mlp_03,132415,164000,0.692375,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
decision_tree_04,117573,164000,0.549625,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
logistic_regression_05,132848,164000,0.695375,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
xgboost_06,132190,164000,0.704125,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
random_forest_07,138931,164000,0.77925,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
mlp_08,132581,164000,0.6985,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."


,records,members,accuracy,balanced_accuracy,roc_auc,tpr_at_1pct_fpr_roc,threshold,tn,fp,fn,tp,shadow_selected_threshold,tpr_at_shadow_threshold,fpr_at_shadow_threshold
train,19279.0,9614.0,0.552518,0.553476,0.650067,0.036093,0.44842,1850.0,7815.0,812.0,8802.0,0.617857,0.015706,0.003518
validation,6319.0,3150.0,0.512423,0.513518,0.503106,0.009524,0.44842,473.0,2696.0,385.0,2765.0,0.617857,0.009524,0.009467
test,6402.0,3236.0,0.516557,0.512576,0.517551,0.006180,0.44842,470.0,2696.0,399.0,2837.0,0.617857,0.008653,0.012003


In [9]:
# Query this notebook's overfit target using its own fitted preprocessor.
target_attack_data, rf_target_walk_diagnostics = collect_target_paper_features(
    overfit_model, overfit_preprocessor, overfit_partitions, config=WALK_CONFIG, seed=42)
display(pd.Series(rf_target_walk_diagnostics, name='target query diagnostics'))
target_attack_metrics, target_membership_scores = evaluate_attack(
    attack_result['classifier'], target_attack_data,
    threshold=attack_result['accuracy_threshold'],
    low_fpr_threshold=attack_result['low_fpr_threshold'])
pd.DataFrame({
    'Adult attack test': attack_result['metrics']['test'],
    'Overfit ACS 2023 target MIA': target_attack_metrics,
}).T

Attack feature extraction: 512/20,000


Attack feature extraction: 1,024/20,000


Attack feature extraction: 1,536/20,000


Attack feature extraction: 2,048/20,000


Attack feature extraction: 2,560/20,000


Attack feature extraction: 3,072/20,000


... (32 outputs left out)

Attack feature extraction: 19,968/20,000


Attack feature extraction: 20,000/20,000


record_queries                                                       519217
max_record_queries                                                   820000
walk_censored_fraction                                               0.4322
blind_walk_enabled                                                     True
config                    {'max_steps': 20, 'repeats': 2, 'categorical_c...
Name: target query diagnostics, dtype: object

,records,members,accuracy,balanced_accuracy,roc_auc,tpr_at_1pct_fpr_roc,threshold,tn,fp,fn,tp,shadow_selected_threshold,tpr_at_shadow_threshold,fpr_at_shadow_threshold
Adult attack test,6402.0,3236.0,0.516557,0.512576,0.517551,0.00618,0.44842,470.0,2696.0,399.0,2837.0,0.617857,0.008653,0.012003
Overfit ACS 2023 target MIA,20000.0,10000.0,0.601650,0.601650,0.625639,0.00490,0.44842,2154.0,7846.0,121.0,9879.0,0.617857,0.001700,0.000800


## Separate ACS2014 auxiliary baseline
Use California ACS2014 records only for this auxiliary attack. Preserve the same
five algorithm families and eight shadows, using the same settings and seed sequence as the Adult baseline. Native RELP
is retained for these shadows; the target uses native RELSHIPP. Each fitted encoder
is local to its own training records. Category codes do not pass between models.
The common representation transferred to the attack is the four query/output features.


In [10]:
acs_2014_auxiliary=load_baseline_year(2014)
ACS_2014_FEATURES=features_for_year(2014)
assert len(ACS_2014_FEATURES)==10
assert set(acs_2014_auxiliary.index).isdisjoint(overfit_source.index)
acs_2014_shadows=train_acs_shadows(acs_2014_auxiliary,ACS_2014_FEATURES,seed=OVERFIT_SEED,n_models=N_SHADOW_MODELS)
assert len(acs_2014_shadows)==N_SHADOW_MODELS
from src.paper_attack_features import make_probability_oracle
for name,result in acs_2014_shadows.items():
    rows=result['attack_data'].iloc[:32]
    query=make_probability_oracle(result['model'],result['preprocessor'],schema='acs')
    np.testing.assert_allclose(query(acs_2014_auxiliary.loc[rows.record_id,ACS_2014_FEATURES]),
        rows[['p_income_le_50k','p_income_gt_50k']].to_numpy(),rtol=1e-5,atol=1e-6)
display(summarize_shadow_models(acs_2014_shadows))


Training ACS shadow 1/8: xgboost_01


Training ACS shadow 2/8: random_forest_02


Training ACS shadow 3/8: mlp_03


Epoch 001: train loss=0.4143, validation loss=0.3776, validation accuracy=0.8258


Epoch 002: train loss=0.3691, validation loss=0.3672, validation accuracy=0.8314


Epoch 003: train loss=0.3603, validation loss=0.3642, validation accuracy=0.8330


... (15 outputs left out)

Epoch 006: train loss=0.3372, validation loss=0.3753, validation accuracy=0.8285


Epoch 007: train loss=0.3317, validation loss=0.3736, validation accuracy=0.8289


Epoch 008: train loss=0.3253, validation loss=0.3775, validation accuracy=0.8292


records  accuracy   roc_auc  log_loss
shadow_model           partition                                        
xgboost_01             train         73576  0.829306  0.907009  0.371119
                       validation    36789  0.822474  0.899399  0.382714
                       nonmembers    73576  0.822034  0.897887  0.384960
random_forest_02       train         73576  0.853594  0.933065  0.351895
                       validation    36789  0.825437  0.900334  0.394538
                       nonmembers    73576  0.822129  0.898750  0.396072
mlp_03                 train         73576  0.843563  0.916767  0.348910
                       validation    36789  0.832994  0.908552  0.364231
                       nonmembers    73576  0.833601  0.908693  0.363665
decision_tree_04       train         73576  0.831168  0.907213  0.363991
                       validation    36789  0.797711  0.857926  0.831248
                       nonmembers    73576  0.794920  0.858373  0.839791
logistic_regression_05 train         73576  0.831100  0.904563  0.372566
                       validation    36789  0.826932  0.900413  0.380331
                       nonmembers    73576  0.825582  0.898124  0.384226
xgboost_06             train         73576  0.829075  0.906883  0.371038
                       validation    36789  0.822284  0.897779  0.385122
                       nonmembers    73576  0.822102  0.898850  0.383457
random_forest_07       train         73576  0.854559  0.933490  0.350271
                       validation    36789  0.819810  0.897958  0.396213
                       nonmembers    73576  0.819683  0.897134  0.397962
mlp_08                 train         73576  0.845928  0.919603  0.344612
                       validation    36789  0.830547  0.905897  0.369117
                       nonmembers    73576  0.831983  0.907460  0.366092

## Fit the ACS2014 attack and compare against the same target queries
Both attacks select thresholds using their own auxiliary validation records only.
Reuse target_attack_data from the Adult baseline to keep target candidates and
queries identical. Target membership labels do not fit either classifier or select settings.


In [11]:
acs_2014_attack_data,acs_2014_query_diagnostics=collect_shadow_paper_features(
    acs_2014_shadows,acs_2014_auxiliary,per_class=SHADOW_RECORDS_PER_CLASS,
    config=WALK_CONFIG,seed=OVERFIT_SEED)
acs_2014_attack=train_membership_classifier(acs_2014_attack_data,seed=OVERFIT_SEED,
    algorithm='random_forest',feature_set='paper')
acs_2014_target_metrics,acs_2014_target_scores=evaluate_attack(
    acs_2014_attack['classifier'],target_attack_data,
    threshold=acs_2014_attack['accuracy_threshold'],
    low_fpr_threshold=acs_2014_attack['low_fpr_threshold'])
baseline_comparison=pd.DataFrame({'UCI Adult auxiliary':target_attack_metrics,
    'ACS2014 auxiliary':acs_2014_target_metrics}).T
display(baseline_comparison[['accuracy','balanced_accuracy','roc_auc','tpr_at_1pct_fpr_roc',
    'tpr_at_shadow_threshold','fpr_at_shadow_threshold']])


Attack feature extraction: 512/4,000


Attack feature extraction: 1,024/4,000


Attack feature extraction: 1,536/4,000


Attack feature extraction: 2,048/4,000


Attack feature extraction: 2,560/4,000


Attack feature extraction: 3,072/4,000


... (55 outputs left out)

Attack feature extraction: 3,072/4,000


Attack feature extraction: 3,584/4,000


Attack feature extraction: 4,000/4,000


,accuracy,balanced_accuracy,roc_auc,tpr_at_1pct_fpr_roc,tpr_at_shadow_threshold,fpr_at_shadow_threshold
UCI Adult auxiliary,0.60165,0.60165,0.625639,0.0049,0.0017,0.0008
ACS2014 auxiliary,0.54315,0.54315,0.573493,0.0070,0.0020,0.0030


## Save both baselines and exact target record assignments

In [12]:
baseline_comparison.to_csv(RUN_DIR/'baseline_comparison.csv')
summarize_shadow_models(shadow_results).to_csv(RUN_DIR/'adult_shadow_metrics.csv')
summarize_shadow_models(acs_2014_shadows).to_csv(RUN_DIR/'acs2014_shadow_metrics.csv')
for name,part in overfit_partitions.items():
    pd.Series(part.index,name='record_id').to_csv(RUN_DIR/(name+'_ids.csv'),index=False)
joblib.dump(dict(adult=attack_result,acs2014=acs_2014_attack),RUN_DIR/'attack_classifiers.joblib')
joblib.dump(dict(features=target_attack_data,adult_scores=target_membership_scores,
    acs2014_scores=acs_2014_target_scores),RUN_DIR/'target_mia_outputs.joblib')
summary=dict(execution_status='all_cells_completed',target_year=2023,target_train_size=OVERFIT_TRAIN_SIZE,
    target_features=TARGET_FEATURES,acs2014_features=ACS_2014_FEATURES,
    relationship_schema='Native RELSHIPP for 2023; native RELP for 2014; separate encodings',
    target_label='nominal PINCP > 50000',target_training=overfit_metadata,target_metrics=overfit_metrics,
    n_shadows_per_auxiliary=N_SHADOW_MODELS,shadow_algorithms=['xgboost','random_forest','mlp','decision_tree','logistic_regression'],
    shadow_records_per_membership=SHADOW_RECORDS_PER_CLASS,walk_config=WALK_CONFIG,
    split_config=SPLIT_CONFIG,adult_target_mia=target_attack_metrics,acs2014_target_mia=acs_2014_target_metrics,
    adult_auxiliary_attack_metrics=attack_result['metrics'],acs2014_auxiliary_attack_metrics=acs_2014_attack['metrics'])
(RUN_DIR/'summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
print('Both baselines saved to',RUN_DIR)


Both baselines saved to artifacts\exp2_2023_adult_2014_baseline_8shadows
